# 1. Building the project and the hydrological model

Workflow: **`build_model.smk`** (WF1)

**Source only; current code requires review.** Outputs are not committed.

Some cells use older configuration or output interfaces. This prose pass does not repair or
execute them. Read the [notebook status](README.md#current-status) before running cells.
Use the [quick start](../guide/quick-start.md) for the maintained command-line route.
Input data are external; a blank source is not a supplied result.


## What this workflow does

WF1 turns a point on a map into a runnable, forced Wflow-SBM model, plus a first
read on how well that model reproduces observed discharge. In order:

1. **Delineates the basin** from the region you specify, along with the
   subcatchments and gauge locations inside it.
2. **Extracts a historical climate store** for that basin from a global dataset
   (ERA5 by default), and plots it.
3. **Prepares the thematic maps** &mdash; land cover, leaf area index, soil
   properties &mdash; that the model is parameterised from.
4. **Builds the Wflow-SBM model** with hydromt, adds reservoirs, lakes and
   glaciers, and declares which state variables to write out.
5. **Adds climate forcing** for the simulation window and **runs Wflow once**.
6. **Evaluates the run** against observed discharge and plots the result.

This is a *rapid deployment*: the model is parameterised from global data and is
not locally calibrated. The evaluation step is what tells you how far to trust
it &mdash; a question worth settling before the stress test in notebook 3, since
that notebook inherits whatever this one produces.

Nothing here is scenario-driven. The perturbations come later; WF1 builds the
reference they are applied to.

## Setup

In [ ]:
# Standard library, plus the few third-party names the results cells need.
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from IPython import display

In [ ]:
# Work from the repository root.
#
# Snakemake resolves every path in a config relative to the directory it is
# invoked from, and the shipped configs are written against the repo root.
# Rather than hardcoding an install path -- which is what made the previous
# version of this notebook unrunnable for anyone but its author -- walk up from
# wherever the kernel started until the Snakefiles come into view.
def find_repo_root(marker: str = "build_model.smk") -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"could not find {marker} at or above {Path.cwd()}; "
        "start the kernel inside the blueearth_cst checkout"
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print(REPO_ROOT)

In [ ]:
# Snakemake runs take minutes, so stream the output rather than buffering it to
# the end of the cell.
#
# `check` raises on a nonzero exit. Without it a failed Snakemake call printed
# its error into the cell output and the cell still rendered as SUCCESSFUL, so
# the first symptom was a later `display.Image` raising on a file that was never
# written -- an error pointing at the wrong place, in a document whose reader is
# the least equipped to tell the difference. Pass `check=False` where a nonzero
# exit is expected, as `snakemake --unlock` gives when there is no lock.
def run(command: str, *, check: bool = True) -> int:
    print("$ " + command)
    print()
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        shell=True,
        stderr=subprocess.STDOUT,
        bufsize=1,
        close_fds=True,
    ) as process:
        for line in iter(process.stdout.readline, b""):
            print(line.rstrip().decode("utf-8", errors="replace"))
    if check and process.returncode != 0:
        raise RuntimeError(
            f"command failed with exit code {process.returncode}: {command}"
        )
    return process.returncode

## The settings file

Everything a run does is decided by one YAML file passed as `--configfile`. The
toolbox ships seed configs beside the project they write into, under
`test_case/`:

| File | Project | Use it for |
|---|---|---|
| `project_config_rapid.yml` | `test_case/test_rapid` | anything you want to watch execute &mdash; the default |
| `project_config_baseline.yml` | `test_case/test_local` | runs whose numbers get quoted or fingerprinted |
| `project_config_wf2_fast.yml` | `test_case/test_dev` | WF2 code iteration only |

To start your own project, copy `config/templates/example-basin/` and follow its README.
If you keep configs under `test_case/`, retain the `project_config_` filename prefix:
the repository's `.gitignore` un-ignores exactly that glob, and a name outside it
is silently untracked.

This notebook runs the rapid config.

In [ ]:
# The settings file this notebook runs against, and the paths derived from it
# that every later cell needs.
CONFIG = "test_case/project_config_rapid.yml"

cfg = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
PROJECT_DIR = Path(cfg["project"]["project_dir"])
PROJECT_NAME = cfg.get("project_name") or PROJECT_DIR.name
print("config      :", CONFIG)
print("project_dir :", PROJECT_DIR)

In [ ]:
# The settings file, in full. It is commented in place -- those comments are the
# documentation for each key, which is why this cell prints the shipped file
# rather than re-authoring a copy that could drift away from it.
print(Path(CONFIG).read_text(encoding="utf-8"))

### Reading the settings file

The project file owns shared basin, climate and model choices. Its workflow entries name
separate settings files with `config_path`. Some cells below still read an older nested
configuration directly and need code repair before use with the current starter.
See [Creating a project](../guide/new-case.md) and [WF1 reference](../technical-manual/workflow-build-model.md).


## Input file formats

Two optional CSVs shape what WF1 produces. Both are shown here rather than
specified, because a schema is easier to copy than to read.

**`basin.output_locations`** &mdash; the locations you want output at.
Required columns are `station_name`, `x`, `y`. Optional: `location_role`
(`control` or `observation`, default `control`) and `wflow_id`. Points are
snapped to the river network during delineation, so a coordinate that sits
slightly off the channel still works.

In [ ]:
pd.read_csv(cfg["basin"]["output_locations"])

**`workflows.build_model.observations`** &mdash; observed
discharge in m&sup3;/s. **Semicolon-separated**, with a `time` column and one
column per station.

The column headers are **`wflow_id` values, not station names**. Those ids are
assigned during the model build, so the sequence is: run once, read the ids back
from the location registry (shown in the results section), then name your
observation columns to match.

In [ ]:
obs = pd.read_csv(
    cfg["workflows"]["build_model"]["observations"],
    sep=";",
    index_col=0,
    parse_dates=True,
)
print(obs.shape)
obs.head()

## The job graph

Preview through the current owned runner before execution.
A missing or invalid setting can be refused; checkpoint-dependent previews may be partial.


In [ ]:
# Render the job graph. `scripts/plot_workflow_dag.py` wraps `snakemake --dag`
# and pipes it through graphviz, so the image lands inside the project's own
# logs/ rather than in whatever directory the shell happened to be in:
#
#     <project_dir>/logs/dag/<project_name>_wf<N>[_<experiment>]_dag.png
#
# Reading the graph before running is the cheapest way to see whether the config
# was understood the way you meant it.
run(f"python scripts/plot_workflow_dag.py -s build_model.smk --configfile {CONFIG}")

In [ ]:
DAG_PNG = PROJECT_DIR / "logs" / "dag" / f"{PROJECT_NAME}_wf1_dag.png"
display.Image(str(DAG_PNG))

### The rules, and the config keys that tune them

Use the [WF1 reference](../technical-manual/workflow-build-model.md) for current rule ownership
and [Creating a project](../guide/new-case.md) for files to edit.
Historical source diagnostics belong to WF0; WF1 builds/evaluates the historical model.


## Running

Clear a stale lock only after confirming no workflow is active.
The following code is retained for review; prefer the [owned runner](../guide/running.md).


In [ ]:
run(f"snakemake --unlock -s build_model.smk --configfile {CONFIG}", check=False)

A dry run lists the jobs that *would* execute, and why, without running
them. On a project that is already up to date it reports nothing to do &mdash;
which is itself the answer to "did my config edit invalidate anything?".

In [ ]:
run(f"snakemake all -n -s build_model.smk --configfile {CONFIG}")

And the real run. `-c 3` gives Snakemake three cores to schedule with;
raise it if you have them, since several branches of this graph are independent.

Expect this to take a while on a cold project &mdash; the hydromt build and the
Wflow run dominate the wall clock.

In [ ]:
run(f"snakemake all -c 3 -s build_model.smk --configfile {CONFIG}")

## Results

In [ ]:
# What the run wrote. Noisy internal branches are skipped so the shape of the
# tree stays readable.
def show_tree(root: Path, skip=(".snakemake", "_parts", "_work"), max_files=8):
    root = Path(root)
    for path, dirs, files in os.walk(root):
        dirs[:] = sorted(d for d in dirs if d not in skip)
        rel = Path(path).relative_to(root)
        print(root.name + "/" if str(rel) == "." else f"{root.name}/{rel}")
        shown = sorted(f for f in files if not f.endswith(".xml"))
        for name in shown[:max_files]:
            print("  - " + name)
        if len(shown) > max_files:
            print(f"  ... {len(shown) - max_files} more")


show_tree(PROJECT_DIR)

### The basin

`elevation_basin.png` is the orientation figure: catchment outline, river network,
elevation, gauge locations, and a locator inset.

Check three things here before looking at any number &mdash; that the outline is
the basin you meant, that the gauge points sit *on* the river rather than beside
it, and that the drainage network is connected. A point that snapped to the
wrong tributary produces a model that runs perfectly well and answers a
different question than the one you asked.

In [ ]:
display.Image(str(PROJECT_DIR / "data" / "spatial" / "plots" / "elevation_basin.png"))

The thematic maps beside it &mdash; land cover, soil texture, depth to
bedrock, leaf area index &mdash; are what Wflow-SBM is parameterised from. They
are worth a glance mainly as a check on the data catalog: a soil map that is
uniform across the basin usually means the global product has no real resolution
here, and the model's spatial variability is coming from topography alone.

In [ ]:
display.Image(str(PROJECT_DIR / "data" / "spatial" / "plots" / "land_cover.png"))

### The forcing

Precipitation, temperature and PET as the model actually saw them. The annual
series is the quickest check that the climate store covers the simulation window
without gaps.

In [ ]:
FORCING_PLOTS = PROJECT_DIR / "models" / "hydrology" / "wflow" / "forcing" / "plots"
display.display(
    display.Image(str(FORCING_PLOTS / "forcing_precip_annual.png")),
    display.Image(str(FORCING_PLOTS / "forcing_temp_annual.png")),
)

### Station identity

The location registry is the bridge between the names you supplied in
`output_locations` and the `wflow_id`s the model assigned &mdash; the ids your
observation file has to use. It also records where each point *snapped to*,
which is the first thing to check when one station's hydrograph looks wrong.

In [ ]:
registry = pd.read_csv(PROJECT_DIR / "data" / "spatial" / "location_registry.csv")
registry[
    [
        "station_name",
        "wflow_id",
        "location_role",
        "original_x",
        "original_y",
        "snapped_x",
        "snapped_y",
    ]
]

### Model performance

`performance_indicators.csv` holds KGE, NSE and their variants per station, at
daily and monthly aggregation.

In [ ]:
indicators = pd.read_csv(
    PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "performance_indicators.csv"
)
indicators.round(3)

**Historical interpretation note, not qualified guidance:** the threshold advice and
rendered-run statements below are retained for later scientific review. They do not establish
fitness for your basin or decision.

**How to read this.** These are uncalibrated results from global data, so
the question is not "is it good" but "is it good enough for the decision this
assessment supports".

- **KGE above ~0.7** means water balance, variability and timing are all roughly
  right at that station. Stress-test results there are worth interpreting in
  absolute terms.
- **KGE ~0.4&ndash;0.7** usually means one component is off, most often the
  mean &mdash; a hydrograph with the right shape at the wrong level. *Relative*
  changes across the stress-test grid can still be informative when absolute
  values are not.
- **KGE below ~0.4** is a station to be careful with. In this rendered run the
  smallest tributary sits here, which is the usual pattern: global forcing on a
  ~1 km grid resolves the main stem far better than a headwater.
- **Monthly better than daily** is normal, and means the model has the seasonal
  cycle but not individual events. For a stress test built on annual and
  seasonal indicators that is often acceptable; for a flood application it is
  not.

Read the indicators *per station*. A good outlet number can hide a tributary that is
not being simulated at all, and averaging across stations is exactly how that
gets missed.

The hydrographs make the same point visually &mdash; one per station,
named by `wflow_id`, simulated against observed.

In [ ]:
hydrographs = sorted(
    (PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "plots").glob(
        "hydrograph_*.png"
    )
)
display.display(*[display.Image(str(p)) for p in hydrographs])

### Provenance

Inspect the model's retained metadata before reuse in WF4.
WF3 weather generation has no model dependency. The following metadata-read cell may
use an older layout; see [retained handoffs](../technical-manual/retained-handoffs.md).


In [ ]:
run_metadata = json.loads(
    (
        PROJECT_DIR / "models" / "hydrology" / "wflow" / "evaluation" / "run_metadata.json"
    ).read_text(encoding="utf-8")
)
print(json.dumps(run_metadata, indent=2))

## Next

[Projection analysis](<Climate projections.ipynb>) is optional plausibility context.
[The stress-test notebook](<Climate Stress Test.ipynb>) explores retained scenarios/responses.
WF1 and WF3 independently supply WF4; generation does not require a model.
Read each notebook's status note before executing its retained code.
